Integrantes:

* Daniel Barbosa Alves - 1845274
* Allan Reis da Conceição - 153960
* Giovana Alves Duarte de Sena - 2594141
* Gabriel Noriler Souza - 1642714



# Aula Prática 01 — MetrôBot SP
### Busca (BFS e DFS) + Lógica Proposicional e de Primeira Ordem + Llama

Este notebook contém o código completo e estruturado da prática do MetrôBot SP para a Linha 1-Azul.

**Regra de ouro:** *O LLM conversa, o algoritmo decide.*

## 1. Instalação das Dependências
> **Explicação do passo:** Instala todas as bibliotecas necessárias no ambiente: cliente do Groq (para a API na nuvem), Ollama (caso use local), `ipywidgets` (para botões e interface interativa) e `python-dotenv`.

In [1]:
# Célula 1 — Instalação (rode uma vez)
%pip install -q groq ollama ipywidgets python-dotenv
!apt-get install -y zstd # Install zstd dependency
!apt-get install -y lspci
!apt-get install -y lshw
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in the background
import subprocess
import time

print("Starting Ollama server...")
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5) # Give some time for the server to start
print("Ollama server started (hopefully).")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 129.7 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 52 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 2s (376 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
Reading package lists... Done
Building dependency tree... Done
R

## 2. Configuração e Função Única do LLM
> **Explicação do passo:** Centraliza a comunicação com o Llama na função `chamar_llm()`. A chave da API é lida de forma segura via **Secrets** do Colab (`userdata.get('GROQ_API_KEY')`), garantindo que nenhuma credencial fique exposta no código. Caso não queira usar API, é possível definir `PROVEDOR = "offline"`.

In [2]:
import os, json, re, unicodedata, ollama # Moved ollama import here for global use
from collections import deque
from itertools import product

PROVEDOR = "ollama"  # "groq" (nuvem), "ollama" (local) ou "offline" (sem LLM)
MODELO_GROQ = "qwen/qwen3.8-27b"  # ou confira em console.groq.com/docs/models
MODELO_OLLAMA = "qwen2.5:3b"

# Added logic to download Ollama model if PROVEDOR is 'ollama'
if PROVEDOR == "ollama":
    print(f"Tentando baixar o modelo Ollama: {MODELO_OLLAMA}...")
    try:
        ollama.pull(MODELO_OLLAMA)
        print("Download do modelo Ollama concluído com sucesso.")
    except Exception as e:
        print(f"Erro ao baixar o modelo Ollama: {e}. Verifique se o Ollama está instalado e em execução.")

def obter_chave_groq():
    """Busca a chave SEM escrevê-la no código: Colab Secrets, .env ou variável de ambiente."""
    try:
        from google.colab import userdata
        return userdata.get("GROQ_API_KEY")
    except Exception:
        pass
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except Exception:
        pass
    return os.environ.get("GROQ_API_KEY")

def chamar_llm(mensagens, modo_json=False):
    """Envia mensagens ao Llama e devolve o texto da resposta."""
    if PROVEDOR == "groq":
        from groq import Groq
        cliente = Groq(api_key=obter_chave_groq())
        extras = {"response_format": {"type": "json_object"}} if modo_json else {}
        resposta = cliente.chat.completions.create(
            model=MODELO_GROQ,
            messages=mensagens,
            temperature=0,
            **extras
        )
        return resposta.choices[0].message.content
    elif PROVEDOR == "ollama":
        extras = {"format": "json"} if modo_json else {}
        resposta = ollama.chat(
            model=MODELO_OLLAMA,
            messages=mensagens,
            options={"temperature": 0},
            **extras
        )
        return resposta["message"]["content"]
    else:
        raise RuntimeError("Modo offline: nenhum LLM configurado.")

Tentando baixar o modelo Ollama: qwen2.5:3b...
Download do modelo Ollama concluído com sucesso.


## 3. Teste de Conexão com o LLM
> **Explicação do passo:** Envia uma mensagem simples para validar o funcionamento do Groq/Llama.

In [3]:
# Célula 3 — Teste de conexão
resposta = chamar_llm([
    {"role": "user", "content": "Em uma frase curta, dê boas-vindas aos passageiros do metrô de São Paulo."}
])
print(resposta)

Bem-vindos ao metrô de São Paulo!


## 4. Modelagem da Linha 1-Azul como Grafo
> **Explicação do passo:** Constrói um grafo representado por lista de adjacência a partir da sequência das 23 estações da Linha 1-Azul.

In [4]:
# Célula 4 — A Linha 1-Azul como grafo
LINHA_1_AZUL = [
    "Tucuruvi", "Parada Inglesa", "Jardim São Paulo", "Santana",
    "Carandiru", "Portuguesa-Tietê", "Armênia", "Tiradentes", "Luz",
    "São Bento", "Sé", "Japão-Liberdade", "São Joaquim", "Vergueiro",
    "Paraíso", "Ana Rosa", "Vila Mariana", "Santa Cruz",
    "Praça da Árvore", "Saúde", "São Judas", "Conceição", "Jabaquara"
]
LINHA_2_VERDE = [
    "Vila Madalena", "Sumaré", "Clínicas", "Consolação", "Trianon-Masp",
    "Brigadeiro", "Paraíso", "Ana Rosa", "Chácara Klabin", "Santos-Imigrantes",
    "Alto do Ipiranga", "Sacomã", "Tamanduateí", "Vila Prudente"
]
TODAS_ESTACOES = list(set(LINHA_1_AZUL + LINHA_2_VERDE))

def construir_grafo(estacoes):
    """Cada estação vira um nó ligado à anterior e à próxima da lista."""
    grafo = {}
    for linha in estacoes:
        # Inicializa a chave se a estação ainda não existir
        for estacao in linha:
            if estacao not in grafo:
                grafo[estacao] = []
        # Conecta os vizinhos
        for i in range(len(linha) - 1):
            a, b = linha[i], linha[i + 1]
            if b not in grafo[a]: grafo[a].append(b)
            if a not in grafo[b]: grafo[b].append(a)
    return grafo

GRAFO = construir_grafo([LINHA_1_AZUL, LINHA_2_VERDE])

## 5. Inspecionando o Grafo
> **Explicação do passo:** Exibe a quantidade de estações e a lista de adjacência das estações exploradas.

In [5]:
# Célula 5 — Explorando o grafo
print("Total de estações:", len(GRAFO))
print("Vizinhas da Sé:", GRAFO["Sé"])
print("Vizinhas do Tucuruvi:", GRAFO["Tucuruvi"])

for estacao, vizinhas in list(GRAFO.items())[:5]:
    print(f"{estacao:>18} -> {vizinhas}")

Total de estações: 35
Vizinhas da Sé: ['São Bento', 'Japão-Liberdade']
Vizinhas do Tucuruvi: ['Parada Inglesa']
          Tucuruvi -> ['Parada Inglesa']
    Parada Inglesa -> ['Tucuruvi', 'Jardim São Paulo']
  Jardim São Paulo -> ['Parada Inglesa', 'Santana']
           Santana -> ['Jardim São Paulo', 'Carandiru']
         Carandiru -> ['Santana', 'Portuguesa-Tietê']


## 6. Locais Conhecidos (Pontos de Interesse)
> **Explicação do passo:** Mapeia atrações e locais de interesse da cidade para a estação de metrô correspondente.

In [6]:
# Célula 6 — Locais conhecidos (local -> estação mais próxima)
LOCAIS = {
    "Shopping Metrô Tucuruvi": "Tucuruvi",
    "Terminal Rodoviário Tietê": "Portuguesa-Tietê",
    "Museu de Arte Sacra": "Tiradentes",
    "Pinacoteca": "Luz",
    "Museu da Língua Portuguesa": "Luz",
    "Mosteiro de São Bento": "São Bento",
    "Rua 25 de Março": "São Bento",
    "Catedral da Sé": "Sé",
    "Bairro da Liberdade": "Japão-Liberdade",
    "Centro Cultural São Paulo": "Vergueiro",
    "Shopping Metrô Santa Cruz": "Santa Cruz",
    "Universidade São Judas": "São Judas",
    "Terminal Rodoviário Jabaquara": "Jabaquara",
    "MASP": "Trianon-Masp",
    "Avenida Paulista": "Consolação",
    "Beco do Batman": "Vila Madalena",
    "Parque da Independência": "Alto do Ipiranga",
    "Museu do Ipiranga": "Alto do Ipiranga",
    "Teatro Municipal": "Vila Prudente"
}

## 7. BFS Passo a Passo (Didática)
> **Explicação do passo:** Mostra em detalhes a BFS atuando com fila (FIFO) e construindo o dicionário `pai`.

In [7]:
# Célula 7 — BFS passo a passo
def bfs_passo_a_passo(grafo, origem, destino):
    fila = deque([origem])
    pai = {origem: None}
    passo = 0
    while fila:
        passo += 1
        atual = fila.popleft()
        if atual == destino:
            print(f"Passo {passo}: visito {atual:<17} -> Cheguei!")
            return pai
        for vizinho in grafo[atual]:
            if vizinho not in pai:
                pai[vizinho] = atual
                fila.append(vizinho)
        print(f"Passo {passo}: visito {atual:<17} fila: {list(fila)}")
    return pai

pai = bfs_passo_a_passo(GRAFO, "Sé", "São Joaquim")
print("\nDicionário pai:", pai)

Passo 1: visito Sé                fila: ['São Bento', 'Japão-Liberdade']
Passo 2: visito São Bento         fila: ['Japão-Liberdade', 'Luz']
Passo 3: visito Japão-Liberdade   fila: ['Luz', 'São Joaquim']
Passo 4: visito Luz               fila: ['São Joaquim', 'Tiradentes']
Passo 5: visito São Joaquim       -> Cheguei!

Dicionário pai: {'Sé': None, 'São Bento': 'Sé', 'Japão-Liberdade': 'Sé', 'Luz': 'São Bento', 'São Joaquim': 'Japão-Liberdade', 'Tiradentes': 'Luz'}


## 8. BFS Oficial com Tratamento de Bloqueios
> **Explicação do passo:** Implementação oficial da Busca em Largura com reconstrução do trajeto e suporte a estações bloqueadas.

In [8]:
# Célula 8 — BFS (Busca em Largura)
def reconstruir_caminho(pai, destino):
    """Puxa o 'fio de Ariadne' do destino até a origem, depois inverte."""
    caminho = []
    atual = destino
    while atual is not None:
        caminho.append(atual)
        atual = pai[atual]
    return list(reversed(caminho))

def bfs(grafo, origem, destino, bloqueadas=()):
    if origem in bloqueadas or destino in bloqueadas:
        return None, []
    fila = deque([origem])
    pai = {origem: None}
    ordem_visita = []
    while fila:
        atual = fila.popleft()  # 1º da fila sai primeiro (FIFO)
        ordem_visita.append(atual)
        if atual == destino:
            return reconstruir_caminho(pai, destino), ordem_visita
        for vizinho in grafo[atual]:
            if vizinho not in pai and vizinho not in bloqueadas:
                pai[vizinho] = atual
                fila.append(vizinho)
    return None, ordem_visita

## 9. Testando a BFS
> **Explicação do passo:** Executa a BFS padrão e verifica o comportamento em caso de estação bloqueada no trajeto.

In [9]:
# Célula 9 — Testando a BFS
caminho, visitados = bfs(GRAFO, "Luz", "Vergueiro")
print("Caminho:", " -> ".join(caminho))
print("Paradas:", len(caminho) - 1)
print("Estações visitadas pela busca:", len(visitados))

# E se a Sé estiver fechada?
caminho, visitados = bfs(GRAFO, "Luz", "Vergueiro", bloqueadas={"Sé"})
print("\nCom a Sé fechada:", caminho)

Caminho: Luz -> São Bento -> Sé -> Japão-Liberdade -> São Joaquim -> Vergueiro
Paradas: 5
Estações visitadas pela busca: 11

Com a Sé fechada: None


## 10. DFS (Busca em Profundidade)
> **Explicação do passo:** Implementa a busca em profundidade com recursão (pilha de chamadas) e backtracking.

In [10]:
# Célula 10 — DFS (Busca em Profundidade)
def dfs(grafo, origem, destino, bloqueadas=()):
    """Vai fundo no 1º vizinho; se não achar, volta (backtracking) e tenta o próximo."""
    if origem in bloqueadas or destino in bloqueadas:
        return None, []
    visitados = set()
    ordem_visita = []

    def explorar(atual, caminho):
        visitados.add(atual)
        ordem_visita.append(atual)
        if atual == destino:
            return caminho
        for vizinho in grafo[atual]:
            if vizinho not in visitados and vizinho not in bloqueadas:
                resultado = explorar(vizinho, caminho + [vizinho])  # mergulha
                if resultado:
                    return resultado
        return None  # beco sem saída -> volta

    return explorar(origem, [origem]), ordem_visita

## 11. Comparativo: BFS vs. DFS
> **Explicação do passo:** Executa viagens comparativas para registrar o número de estações visitadas por BFS e DFS.

In [11]:
# Célula 11 — Comparando BFS e DFS
viagens = [
    ("Sé", "Japão-Liberdade"),
    ("Sé", "São Joaquim"),
    ("Luz", "Vergueiro"),
    ("Santana", "Sé"),
    ("Paraíso", "Tucuruvi")
]

print(f"{'Viagem':<28} {'Paradas':>8} {'BFS visitou':>13} {'DFS visitou':>13}")
for origem, destino in viagens:
    c_bfs, v_bfs = bfs(GRAFO, origem, destino)
    c_dfs, v_dfs = dfs(GRAFO, origem, destino)
    print(f"{origem + ' -> ' + destino:<28} {len(c_bfs)-1:>8} {len(v_bfs):>13} {len(v_dfs):>13}")

_, v = dfs(GRAFO, "Sé", "Japão-Liberdade")
print("\nOrdem da DFS de Sé até Japão-Liberdade:", v)

Viagem                        Paradas   BFS visitou   DFS visitou
Sé -> Japão-Liberdade               1             3            12
Sé -> São Joaquim                   2             5            13
Luz -> Vergueiro                    5            11            14
Santana -> Sé                       7            11            11
Paraíso -> Tucuruvi                14            35            15

Ordem da DFS de Sé até Japão-Liberdade: ['Sé', 'São Bento', 'Luz', 'Tiradentes', 'Armênia', 'Portuguesa-Tietê', 'Carandiru', 'Santana', 'Jardim São Paulo', 'Parada Inglesa', 'Tucuruvi', 'Japão-Liberdade']


## 12. Lógica Proposicional: Tabela-Verdade
> **Explicação do passo:** Implementa a verificação da regra $P \land (\neg Q \lor R)$ e gera a tabela-verdade completa.

In [12]:
# Célula 12 — Lógica proposicional
def pode_embarcar(P, Q, R):
    """P: estação aberta | Q: precisa de acessibilidade | R: elevador funcionando"""
    return P and ((not Q) or R)

def tabela_verdade():
    print(f"{'P':<5} {'Q':<5} {'R':<5} | {'P ^ (~Q v R)':<12}")
    print("-" * 35)
    for P, Q, R in product([True, False], repeat=3):
        print(f"{str(P):<5} {str(Q):<5} {str(R):<5} | {str(pode_embarcar(P, Q, R)):<12}")

tabela_verdade()

P     Q     R     | P ^ (~Q v R)
-----------------------------------
True  True  True  | True        
True  True  False | False       
True  False True  | True        
True  False False | True        
False True  True  | False       
False True  False | False       
False False True  | False       
False False False | False       


## 13. Lógica de Primeira Ordem: Fatos, Regras e Motor de Inferência
> **Explicação do passo:** Define a base de conhecimento com fatos e regras de produção e implementa o algoritmo de encadeamento para frente (*forward chaining*).

In [13]:
# Célula 13 — Lógica de primeira ordem: fatos, regras e motor de inferência
def fatos_base():
    """Fatos fixos do mundo: quais estações existem e o que fica perto de cada uma."""
    fatos = set()
    for estacao in TODAS_ESTACOES:
        fatos.add(("estacao", estacao))
    for local, estacao in LOCAIS.items():
        fatos.add(("proximo_de", local, estacao))
    return fatos

def consultar(fatos, predicado):
    """Devolve os argumentos de todos os fatos de um predicado. Ex.: consultar(f, 'destino') -> [('Luz',)]"""
    return [f[1:] for f in fatos if f[0] == predicado]

def r_origem(fatos):
    novos = set()
    for (local,) in consultar(fatos, "usuario_esta_em"):
        for (l, e) in consultar(fatos, "proximo_de"):
            if l == local:
                novos.add(("origem", e))
    for (e,) in consultar(fatos, "usuario_esta_na_estacao"):
        novos.add(("origem", e))
    return novos

def r_destino(fatos):
    novos = set()
    for (local,) in consultar(fatos, "usuario_quer_ir"):
        for (l, e) in consultar(fatos, "proximo_de"):
            if l == local:
                novos.add(("destino", e))
    for (e,) in consultar(fatos, "usuario_quer_ir_estacao"):
        novos.add(("destino", e))
    return novos

def r_bloqueio(fatos):
    return {("bloqueada", e) for (e,) in consultar(fatos, "fechada")}

def r_acessibilidade(fatos):
    if not consultar(fatos, "precisa_acessibilidade"):
        return set()
    return {("inacessivel", e) for (e,) in consultar(fatos, "elevador_em_manutencao")}

def r_alerta(fatos):
    novos = set()
    inacessiveis = {e for (e,) in consultar(fatos, "inacessivel")}
    for papel in ("origem", "destino"):
        for (e,) in consultar(fatos, papel):
            if e in inacessiveis:
                novos.add(("alerta", papel, e))
    return novos

REGRAS = [
    ("R1 origem", "∀l ∀e (usuario_esta_em(l) ∧ proximo_de(l,e) → origem(e))", r_origem),
    ("R2 destino", "∀l ∀e (usuario_quer_ir(l) ∧ proximo_de(l,e) → destino(e))", r_destino),
    ("R3 bloqueio", "∀e (fechada(e) → bloqueada(e))", r_bloqueio),
    ("R4 acessibilidade", "∀e (precisa_acessibilidade ∧ elevador_em_manutencao(e) → inacessivel(e))", r_acessibilidade),
    ("R5 alerta", "∀p ∀e (papel(p,e) ∧ inacessivel(e) → alerta(p,e))", r_alerta)
]

def encadear_para_frente(fatos, regras, verbose=False):
    """Aplica as regras em rodadas até não surgir nenhum fato novo (ponto fixo)."""
    fatos = set(fatos)
    justificativas = {}
    rodada = 0
    while True:
        rodada += 1
        novos_na_rodada = set()
        for nome, formula, regra in regras:
            for fato in regra(fatos) - fatos:
                novos_na_rodada.add(fato)
                justificativas[fato] = nome
        if verbose:
            print(f"Rodada {rodada}: {len(novos_na_rodada)} fato(s) novo(s)")
        if not novos_na_rodada:
            return fatos, justificativas
        fatos |= novos_na_rodada

## 14. Efeito Dominó do Motor Lógico
> **Explicação do passo:** Demonstra o processo de inferência lógica iterativa até o ponto fixo.

In [14]:
# Célula 14 — Vendo o dominó acontecer
fatos = fatos_base() | {
    ("usuario_quer_ir", "Pinacoteca"),
    ("precisa_acessibilidade",),
    ("elevador_em_manutencao", "Luz")
}

fatos, justificativas = encadear_para_frente(fatos, REGRAS, verbose=True)

print("\nFatos deduzidos e suas justificativas:")
for fato, regra in sorted(justificativas.items(), key=lambda x: x[1]):
    print(f"  {fato} <- ({regra})")

print("\nConsulta: qual é o destino?", consultar(fatos, "destino"))

Rodada 1: 2 fato(s) novo(s)
Rodada 2: 1 fato(s) novo(s)
Rodada 3: 0 fato(s) novo(s)

Fatos deduzidos e suas justificativas:
  ('destino', 'Luz') <- (R2 destino)
  ('inacessivel', 'Luz') <- (R4 acessibilidade)
  ('alerta', 'destino', 'Luz') <- (R5 alerta)

Consulta: qual é o destino? [('Luz',)]


## 15. Planejador: Integrando Lógica e Busca
> **Explicação do passo:** Função orquestradora que constrói os fatos do cenário, deduz restrições por inferência e aciona a busca em grafos.

In [15]:
# Célula 15 — Planejador (lógica + busca)
TEMPO_POR_TRECHO = 2  # minutos por trecho (valor simulado, didático)

def planejar(pedido, fechadas=(), manutencao=(), algoritmo="BFS"):
    """
    pedido = {"origem": (tipo, nome), "destino": (tipo, nome), "acessibilidade": bool}
    tipo é "local" ou "estacao".
    """
    # 1) Monta a base de conhecimento com o pedido e o cenário
    fatos = fatos_base()
    tipo_o, nome_o = pedido["origem"]
    tipo_d, nome_d = pedido["destino"]

    fatos.add(("usuario_esta_em", nome_o) if tipo_o == "local" else ("usuario_esta_na_estacao", nome_o))
    fatos.add(("usuario_quer_ir", nome_d) if tipo_d == "local" else ("usuario_quer_ir_estacao", nome_d))

    if pedido.get("acessibilidade"):
        fatos.add(("precisa_acessibilidade",))
    for e in fechadas:
        fatos.add(("fechada", e))
    for e in manutencao:
        fatos.add(("elevador_em_manutencao", e))

    # 2) Inferência lógica
    fatos, justificativas = encadear_para_frente(fatos, REGRAS)
    origem = consultar(fatos, "origem")[0][0]
    destino = consultar(fatos, "destino")[0][0]
    bloqueadas = {e for (e,) in consultar(fatos, "bloqueada")}
    alertas = consultar(fatos, "alerta")

    # 3) Busca usando o que a lógica deduziu
    buscar = bfs if algoritmo == "BFS" else dfs
    caminho, visitados = buscar(GRAFO, origem, destino, bloqueadas)

    return {
        "origem": origem,
        "destino": destino,
        "algoritmo": algoritmo,
        "caminho": caminho,
        "visitados": visitados,
        "bloqueadas": sorted(bloqueadas),
        "alertas": [f"{papel}: {e}" for papel, e in alertas],
        "paradas": len(caminho) - 1 if caminho else None,
        "tempo_min": (len(caminho) - 1) * TEMPO_POR_TRECHO if caminho else None,
        "regras_usadas": sorted(set(justificativas.values()))
    }

## 16. Testando o Planejador
> **Explicação do passo:** Executa um teste prático de ponta a ponta das deduções e buscas do planejador.

In [16]:
# Célula 16 — Testando o planejador
r = planejar(
    {"origem": ("local", "Catedral da Sé"), "destino": ("local", "Pinacoteca"), "acessibilidade": True},
    manutencao=["Luz"]
)

for chave, valor in r.items():
    print(f"{chave:>14}: {valor}")

        origem: Sé
       destino: Luz
     algoritmo: BFS
       caminho: ['Sé', 'São Bento', 'Luz']
     visitados: ['Sé', 'São Bento', 'Japão-Liberdade', 'Luz']
    bloqueadas: []
       alertas: ['destino: Luz']
       paradas: 2
     tempo_min: 4
 regras_usadas: ['R1 origem', 'R2 destino', 'R4 acessibilidade', 'R5 alerta']


## 17. Intérprete com Validação e Modo Offline
> **Explicação do passo:** Realiza a extração de intenção para JSON com guardrails para nomes válidos e suporte a fallback offline.

In [17]:
# Célula 17 — Intérprete (Llama) com validação e modo offline
def normalizar(texto):
    """Minúsculas e sem acentos: 'São Bento' -> 'sao bento'."""
    texto = unicodedata.normalize("NFD", texto.lower())
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

def resolver_nome(nome):
    """GUARDRAIL: só aceita nomes que existem de verdade. Senão, None."""
    if not nome:
        return None
    alvo = normalizar(nome).strip()
    for estacao in TODAS_ESTACOES:
        if normalizar(estacao) == alvo:
            return ("estacao", estacao)
    for local in LOCAIS:
        if normalizar(local) == alvo:
            return ("local", local)
    return None

PROMPT_INTERPRETE = """Você é o módulo de INTERPRETAÇÃO do MetrôBot SP.
Sua única tarefa é transformar o pedido do passageiro em JSON.
Estações válidas: {estacoes}
Locais válidos: {locais}
Responda APENAS com um JSON neste formato:
{{"origem": "<nome exato de estação ou local, ou null>", "destino": "<nome exato de estação ou local, ou null>", "acessibilidade": <true ou false>}}
Regras:
- Use SOMENTE nomes das listas acima, escritos exatamente como aparecem.
- acessibilidade é true se o passageiro mencionar cadeira de rodas, mobilidade reduzida, muletas, carrinho de bebê ou precisar de elevador.
- Se não souber algum campo, use null. Nunca invente nomes."""

def interpretar_offline(texto):
    """Plano B sem LLM: procura nomes conhecidos no texto, na ordem em que aparecem."""
    texto_min = texto.lower()
    texto_sem = normalizar(texto)
    candidatos = [(n, "estacao") for n in TODAS_ESTACOES] + [(n, "local") for n in LOCAIS]
    candidatos.sort(key=lambda c: len(c[0]), reverse=True)
    ocupado = [False] * len(texto_min)
    encontrados = []
    for nome, tipo in candidatos:
        buscas = [(texto_min, nome.lower())]
        if len(nome) > 4 and len(texto_sem) == len(texto_min):
            buscas.append((texto_sem, normalizar(nome)))
        for base, padrao in buscas:
            for m in re.finditer(r"(?<!\w)" + re.escape(padrao) + r"(?!\w)", base):
                if not any(ocupado[m.start():m.end()]):
                    encontrados.append((m.start(), nome))
                    for i in range(m.start(), m.end()):
                        ocupado[i] = True
    encontrados.sort()
    palavras_acess = ["cadeira de rodas", "acessibilidade", "mobilidade", "muleta", "carrinho de bebe", "elevador"]
    return {
        "origem": encontrados[0][1] if len(encontrados) > 0 else None,
        "destino": encontrados[1][1] if len(encontrados) > 1 else None,
        "acessibilidade": any(p in texto_sem for p in palavras_acess)
    }

def interpretar_pedido(texto):
    """Texto livre -> pedido validado. Usa o Llama; se falhar, cai no modo offline."""
    if PROVEDOR == "offline":
        bruto = interpretar_offline(texto)
        fonte = "offline"
    else:
        sistema = PROMPT_INTERPRETE.format(
            estacoes=", ".join(TODAS_ESTACOES), locais=", ".join(LOCAIS)
        )
        try:
            resposta = chamar_llm([
                {"role": "system", "content": sistema},
                {"role": "user", "content": texto}
            ], modo_json=True)
            bruto = json.loads(resposta)
            fonte = PROVEDOR
        except Exception as erro:
            print(f"LLM indisponível ({erro}). Usando modo offline.")
            bruto = interpretar_offline(texto)
            fonte = "offline"

    origem = resolver_nome(bruto.get("origem"))
    destino = resolver_nome(bruto.get("destino"))
    if origem is None or destino is None:
        return None, f"Não entendi origem/destino (resposta bruta: {bruto})"

    pedido = {
        "origem": origem,
        "destino": destino,
        "acessibilidade": bool(bruto.get("acessibilidade"))
    }
    return pedido, f"Interpretado via {fonte}"

## 18. Testando o Intérprete
> **Explicação do passo:** Testa a interpretação com frases coloquiais e verifica a rejeição de estações inválidas.

In [18]:
# Célula 18 — Testando o intérprete
pedidos = [
    "Estou na Catedral da Sé e quero ir ao Terminal Rodoviário Jabaquara",
    "to na se, bora pra pinacoteca, tô de cadeira de rodas",
    "Preciso sair do Mosteiro de São Bento e chegar na São Judas",
    "Quero ir da Sé até a Avenida Paulista"
]

for texto in pedidos:
    pedido, mensagem = interpretar_pedido(texto)
    print(">", texto)
    print(" ", mensagem, "->", pedido, "\n")

> Estou na Catedral da Sé e quero ir ao Terminal Rodoviário Jabaquara
  Interpretado via ollama -> {'origem': ('local', 'Catedral da Sé'), 'destino': ('local', 'Terminal Rodoviário Jabaquara'), 'acessibilidade': False} 

> to na se, bora pra pinacoteca, tô de cadeira de rodas
  Interpretado via ollama -> {'origem': ('estacao', 'São Bento'), 'destino': ('local', 'Pinacoteca'), 'acessibilidade': True} 

> Preciso sair do Mosteiro de São Bento e chegar na São Judas
  Interpretado via ollama -> {'origem': ('local', 'Mosteiro de São Bento'), 'destino': ('estacao', 'São Judas'), 'acessibilidade': False} 

> Quero ir da Sé até a Avenida Paulista
  Interpretado via ollama -> {'origem': ('estacao', 'Sé'), 'destino': ('local', 'Avenida Paulista'), 'acessibilidade': False} 



## 19. Módulo Narrador
> **Explicação do passo:** Constrói uma resposta humana e simpática para o usuário baseada exclusivamente no resultado computado.

In [19]:
# Célula 19 — Narrador
def narrar_offline(r):
    if r["caminho"] is None:
        return (f"Não existe rota de {r['origem']} até {r['destino']} "
                f"com as estações bloqueadas: {', '.join(r['bloqueadas'])}.")
    texto = (f"Embarque em {r['origem']} e siga pela Linha 1-Azul até {r['destino']}: "
             f"{r['paradas']} parada(s), cerca de {r['tempo_min']} minutos.")
    if r["alertas"]:
        texto += " Atenção: " + "; ".join(r["alertas"]) + " (elevador em manutenção)."
    return texto

PROMPT_NARRADOR = """Você é o NARRADOR do MetrôBot SP. Explique a rota ao passageiro
em português, em no máximo 4 frases curtas e simpáticas.
Use SOMENTE os dados do JSON. Não invente horários, linhas, estações
ou atrações. Se caminho for null, explique que não há rota e cite as
estações bloqueadas. Se houver 'alertas', destaque-os."""

def narrar(resultado):
    """Transforma o resultado da busca em explicação amigável."""
    dados = {k: resultado[k] for k in ("origem", "destino", "caminho", "paradas", "tempo_min", "bloqueadas", "alertas")}
    if PROVEDOR == "offline":
        return narrar_offline(resultado)
    try:
        return chamar_llm([
            {"role": "system", "content": PROMPT_NARRADOR},
            {"role": "user", "content": json.dumps(dados, ensure_ascii=False)}
        ])
    except Exception as erro:
        return narrar_offline(resultado) + f" (narrador offline: {erro})"

## 20. Teste de Narração
> **Explicação do passo:** Verifica o formato final da resposta gerada pelo narrador.

In [20]:
# Célula 20 — Teste de narração
r = planejar(
    {"origem": ("local", "Catedral da Sé"), "destino": ("local", "Pinacoteca"), "acessibilidade": True},
    manutencao=["Luz"]
)
print(narrar(r))

Sua viagem começa no Sé e vai por São Bento até Luz. São duas paradas, leva aproximadamente 4 minutos. Alerta: o destino Luz está bloqueado. Não há estações bloqueadas.


## 21. Desenho Visual da Linha em HTML
> **Explicação do passo:** Renderiza a linha de metrô com nós coloridos indicando origem, destino, trajeto, estações bloqueadas e esforço da busca.

In [21]:
# Célula 21 — Desenho da Linha 1
def desenhar_linha(resultado):
    caminho = set(resultado["caminho"] or [])
    visitados = set(resultado["visitados"])
    bloqueadas = set(resultado["bloqueadas"])

    html_completo = ""

    # Define as linhas, suas estações e as cores base/destaque (hexadecimal)
    linhas_para_desenhar = [
        ("Linha 1 - Azul", LINHA_1_AZUL, "#1e88e5", "#0d47a1"),
        ("Linha 2 - Verde", LINHA_2_VERDE, "#43a047", "#1b5e20")
    ]

    for nome_linha, estacoes_linha, cor_base, cor_destaque in linhas_para_desenhar:
        # Título da linha
        linhas_html = [f"<div style='margin-bottom: 8px; font-weight: bold; font-family:sans-serif;'>{nome_linha}</div>"]

        for estacao in estacoes_linha:
            if estacao in bloqueadas:
                cor, marca = "#d32f2f", "bloqueada"
            elif estacao in (resultado["origem"], resultado["destino"]) and estacao in caminho:
                cor, marca = cor_destaque, ("origem" if estacao == resultado["origem"] else "destino")
            elif estacao in caminho:
                cor, marca = cor_base, "rota"
            elif estacao in visitados:
                cor, marca = "#9e9e9e", "visitada pela busca"
            else:
                cor, marca = "#e0e0e0", ""

            linhas_html.append(
                f"<div style='display:flex;align-items:center;gap:8px;font-family:sans-serif;font-size:13px'>"
                f"<span style='display:inline-block;width:14px;height:14px;border-radius:50%;background:{cor}'></span>"
                f"<span style='min-width:150px'>{estacao}</span><span style='color:#666'>{marca}</span></div>"
            )

        # Agrupa os nós da linha com uma borda lateral na cor da linha
        html_completo += f"<div style='border-left:4px solid {cor_base};padding-left:8px; margin-bottom: 20px;'>" + "".join(linhas_html) + "</div>"

    return html_completo

## 22. Interface Interativa com ipywidgets
> **Explicação do passo:** Configura os campos, caixas de seleção, botões e callbacks da interface gráfica.

In [22]:
# Célula 22 — Interface com ipywidgets
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Ordena a malha completa alfabeticamente para facilitar a busca nos menus
estacoes_ordenadas = sorted(TODAS_ESTACOES)

# Atualiza a lista de opções substituindo a LINHA_1_AZUL original
opcoes = ([(f"{local} (local)", ("local", local)) for local in LOCAIS] +
          [(f"{estacao} (estação)", ("estacao", estacao)) for estacao in estacoes_ordenadas])

txt_pedido = widgets.Textarea(
    placeholder="Ex.: Estou na Catedral da Sé e quero ir ao MASP",
    layout=widgets.Layout(width="95%", height="60px")
)
btn_interpretar = widgets.Button(description="Interpretar pedido", button_style="info")
dd_origem = widgets.Dropdown(options=opcoes, description="Origem:")
dd_destino = widgets.Dropdown(options=opcoes, value=("local", "Terminal Rodoviário Jabaquara"), description="Destino:")
chk_acess = widgets.Checkbox(description="Preciso de acessibilidade")

# Atualiza os seletores múltiplos para receberem TODAS_ESTACOES em vez de LINHA_1_AZUL
sel_fechadas = widgets.SelectMultiple(options=estacoes_ordenadas, description="Fechadas:", rows=5)
sel_manut = widgets.SelectMultiple(options=estacoes_ordenadas, description="Elevador:", rows=5)

rb_algoritmo = widgets.RadioButtons(options=["BFS", "DFS"], description="Busca:")
btn_buscar = widgets.Button(description="Buscar rota", button_style="success")
saida = widgets.Output()

def ao_interpretar(_):
    with saida:
        clear_output()
        pedido, msg = interpretar_pedido(txt_pedido.value)
        print(msg)
        if pedido:
            dd_origem.value = pedido["origem"]
            dd_destino.value = pedido["destino"]
            chk_acess.value = pedido["acessibilidade"]
            print("Campos preenchidos. Confira e clique em 'Buscar rota'.")

def ao_buscar(_):
    with saida:
        clear_output()
        pedido = {
            "origem": dd_origem.value,
            "destino": dd_destino.value,
            "acessibilidade": chk_acess.value
        }
        r = planejar(pedido, sel_fechadas.value, sel_manut.value, rb_algoritmo.value)
        display(HTML(f"<h4>{r['algoritmo']}: {r['origem']} -> {r['destino']}</h4>"))

        # Narração gerada pelo LLM ou fallback offline
        print("\n", narrar(r))

        # Exibe a rota completa (se existir)
        if r["caminho"]:
            print(f"\nRota encontrada ({len(r['caminho'])} estações):")
            print(" -> ".join(r["caminho"]))
        else:
            print("\nNenhuma rota encontrada.")

        # Exibe todas as estações exploradas pela BFS/DFS
        print(f"\nEstações visitadas pela busca ({len(r['visitados'])}):")
        print(", ".join(r["visitados"]))

        print(f"\nRegras disparadas: {', '.join(r['regras_usadas'])}\n")

        # Renderiza o novo HTML com as duas linhas
        display(HTML(desenhar_linha(r)))

btn_interpretar.on_click(ao_interpretar)
btn_buscar.on_click(ao_buscar)

painel = widgets.VBox([
    # Título do painel atualizado
    widgets.HTML("<h3>🚇 MetrôBot SP — Linhas Azul e Verde</h3>"),
    txt_pedido, btn_interpretar,
    widgets.HBox([dd_origem, dd_destino]),
    widgets.HBox([chk_acess, rb_algoritmo]),
    widgets.HBox([sel_fechadas, sel_manut]),
    btn_buscar, saida
])


## 23. Exibição do Aplicativo
> **Explicação do passo:** Renderiza o painel completo no notebook.

In [23]:
# Célula 23 — Mostrar o app
display(painel)